## Unified Images Notebook

This notebook merges `images.ipynb`, `images_nadavVersion.ipynb`, and `images_nadavVersion2.ipynb` into one non-redundant workflow.

It uses the same path convention as the other updated notebooks:
- Reads from `../../data/...` and `../../output/model/...`
- Saves figures and tables to `../../output/images/`

In [ ]:
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE
from scipy.stats import gaussian_kde

sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

In [ ]:
# Paths aligned with other notebooks in this repo
OUTPUT_ROOT = Path('../../output')
IMG_DIR = OUTPUT_ROOT / 'images'
FIG_DIR = IMG_DIR / 'figures'
TABLE_DIR = IMG_DIR / 'tables'

for p in [IMG_DIR, FIG_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

FIG_SIZE = (8, 5)
K_LABELS = {
    0: 'Original',
    1: 'SAT Rewrite 1',
    2: 'SAT Rewrite 2',
    3: 'SAT Rewrite 3',
    4: 'SAT Rewrite 4',
    5: 'SAT Rewrite 5',
    6: 'SAT Rewrite 6',
    7: 'Standard GPT'
}

print('Saving figures to:', FIG_DIR)
print('Saving tables to :', TABLE_DIR)

In [ ]:
def read_csv_first(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return pd.read_csv(p, encoding='ISO-8859-1', index_col=0)
    raise FileNotFoundError('Could not find any of: ' + ', '.join(map(str, paths)))


def load_npy_first(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return np.load(p)
    raise FileNotFoundError('Could not find any of: ' + ', '.join(map(str, paths)))

### Load data

In [ ]:
df_persuade = read_csv_first([
    '../../data/persuade/persuade_full_cleaned.csv'
])

embeddings_full = load_npy_first([
    '../../data/embeddings/embeddings_full.npy'
])
embeddings_low = load_npy_first([
    '../../data/embeddings/embeddings_low_v2.npy',
    '../../data/embeddings/embeddings_low.npy'
])
embeddings_high = load_npy_first([
    '../../data/embeddings/embeddings_high_v2.npy',
    '../../data/embeddings/embeddings_high.npy'
])

df_sat = read_csv_first([
    '../../data/results/sat_full/data_sat_scored.csv'
]).sort_values(['essay_id', 'k']).reset_index(drop=True)

df_decomp = read_csv_first([
    '../../data/results/sat_full/decomp_rows_with_fe.csv'
]).sort_values(['essay_id', 'k']).reset_index(drop=True)

df_no_desc = read_csv_first([
    '../../data/results/no_desc/no_desc_scored.csv'
]).sort_values(['essay_id', 'k']).reset_index(drop=True)

df_low_scorer = read_csv_first([
    '../../output/model/run_01/data_low_scored.csv',
    '../../model/run_01/data_low_scored.csv'
])
df_high_scorer = read_csv_first([
    '../../output/model/run_01/data_high_scored.csv',
    '../../model/run_01/data_high_scored.csv'
])

print('persuade rows:', len(df_persuade))
print('sat rows     :', len(df_sat))
print('decomp rows  :', len(df_decomp))
print('no_desc rows :', len(df_no_desc))

In [ ]:
# Plain GPT (k=7) if available
try:
    df_plain = read_csv_first([
        '../../data/results/no_desc/no_desc_scored.csv'
    ])
    df_plain = df_plain.copy()
    df_plain['k'] = 7
except Exception:
    df_plain = pd.DataFrame()

if len(df_plain):
    keep_cols = [c for c in df_sat.columns if c in df_plain.columns]
    df_rewrites = pd.concat([df_sat[keep_cols], df_plain[keep_cols]], ignore_index=True)
else:
    df_rewrites = df_sat.copy()

df_rewrites = df_rewrites.sort_values(['essay_id', 'k']).reset_index(drop=True)
print('rewrite table rows:', len(df_rewrites))

### Original dataset visuals (non-redundant set)

In [ ]:
def save_fig(name):
    out = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(out, dpi=300, bbox_inches='tight')
    plt.show()


score_col = 'holistic_essay_score'
ses_col = 'low_SES'

if score_col in df_persuade.columns and ses_col in df_persuade.columns:
    plt.figure(figsize=FIG_SIZE)
    sns.histplot(data=df_persuade, x=score_col, hue=ses_col, bins=20, stat='density', common_norm=False, alpha=0.4)
    plt.xlabel('True essay score')
    plt.ylabel('Density')
    save_fig('real_scores.pdf')
else:
    print('Skipped real_scores: required columns not found.')

In [ ]:
prompt_col = 'prompt_name'
if all(c in df_persuade.columns for c in [prompt_col, score_col, ses_col]):
    tmp = df_persuade.groupby([prompt_col, ses_col])[score_col].mean().reset_index()
    plt.figure(figsize=(10, 5))
    sns.barplot(data=tmp, x=prompt_col, y=score_col, hue=ses_col)
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Prompt')
    plt.ylabel('Mean true score')
    save_fig('mean_score_prompts.pdf')
else:
    print('Skipped mean_score_prompts: required columns not found.')

In [ ]:
# Scatter-matrix summary by SES (single combined cell to avoid repetition)
candidate_cols = [c for c in ['holistic_essay_score', 'word_count', 'sentence_count', 'grade_level'] if c in df_persuade.columns]
if len(candidate_cols) >= 2 and ses_col in df_persuade.columns:
    for ses_val, label in [(0, 'high_ses'), (1, 'low_ses')]:
        sub = df_persuade[df_persuade[ses_col] == ses_val][candidate_cols].dropna()
        if len(sub) > 2:
            g = sns.pairplot(sub.sample(min(3000, len(sub)), random_state=42), corner=True, plot_kws={'s': 8, 'alpha': 0.35})
            g.fig.set_size_inches(8, 8)
            g.fig.savefig(FIG_DIR / f'scatter_matrix_{label}.pdf', dpi=300, bbox_inches='tight')
            plt.show()
else:
    print('Skipped scatter matrices: numeric columns not available.')

### Embeddings visuals

In [ ]:
# Use a bounded sample for speed and stability
max_n = min(12000, len(embeddings_full))
idx = np.random.default_rng(42).choice(len(embeddings_full), size=max_n, replace=False)
X = embeddings_full[idx]

tsne = TSNE(n_components=2, random_state=42, perplexity=30, init='pca')
X_2d = tsne.fit_transform(X)

tsne_df = pd.DataFrame({'x': X_2d[:, 0], 'y': X_2d[:, 1]})
if 'prompt_name' in df_persuade.columns and len(df_persuade) >= len(idx):
    tsne_df['prompt_name'] = df_persuade.iloc[idx]['prompt_name'].values

tsne_df.head()

In [ ]:
plt.figure(figsize=(8, 6))
sns.kdeplot(data=tsne_df, x='x', y='y', fill=True, levels=40, thresh=0.02, cmap='viridis')
sns.scatterplot(data=tsne_df.sample(min(3000, len(tsne_df)), random_state=42), x='x', y='y', s=6, alpha=0.25, color='white', edgecolor=None)
plt.title('t-SNE density (full embeddings)')
save_fig('t_sne_kde.pdf')

In [ ]:
if 'prompt_name' in tsne_df.columns:
    prompts = sorted(tsne_df['prompt_name'].dropna().unique())
    n = len(prompts)
    ncols = 6
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 2.8 * nrows), sharex=True, sharey=True)
    axes = np.array(axes).reshape(-1)

    for i, p in enumerate(prompts):
        ax = axes[i]
        sub = tsne_df[tsne_df['prompt_name'] == p]
        if len(sub) > 5:
            sns.kdeplot(data=sub, x='x', y='y', fill=True, levels=20, thresh=0.03, cmap='mako', ax=ax)
        ax.set_title(str(p), fontsize=9)
        ax.set_xlabel('')
        ax.set_ylabel('')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    fig.tight_layout()
    fig.savefig(FIG_DIR / 'tsne_grid.pdf', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('Skipped tsne_grid: prompt_name unavailable.')

### Predicted-score visuals

In [ ]:
def overlay_hist(df, x_col, hue_col=None, bins=35, title='', fname='plot.pdf'):
    plt.figure(figsize=FIG_SIZE)
    sns.histplot(data=df, x=x_col, hue=hue_col, bins=bins, stat='density', common_norm=False, alpha=0.45)
    plt.title(title)
    plt.xlabel(x_col)
    plt.ylabel('Density')
    save_fig(fname)


for col, fname in [
    ('score_high_full', 'predicted_score_high_scorer.pdf'),
    ('score_low_full', 'predicted_score_low_scorer.pdf'),
]:
    if col in df_rewrites.columns:
        overlay_hist(df_rewrites[df_rewrites['k'] == 0], col, hue_col='low_SES', title=f'{col} at k=0', fname=fname)

In [ ]:
# Feature-specific histograms (one loop to avoid repetitive cells)
for m in ['full', 'style', 'emb', 'taaled', 'taaco', 'taassc']:
    c_high = f'xgb_oof_pred_x_{m}'
    if c_high in df_high_scorer.columns:
        overlay_hist(df_high_scorer, c_high, None, 35, f'High scorer: {m}', f'predicted_score_high_scorer_{m}.pdf')

    c_low = f'xgb_oof_pred_x_{m}'
    if c_low in df_low_scorer.columns:
        overlay_hist(df_low_scorer, c_low, None, 35, f'Low scorer: {m}', f'predicted_score_low_scorer_{m}.pdf')

### Rewrites analysis (k-level differences, CI, distributions)

In [ ]:
rw = df_rewrites.copy()
if 'k' in rw.columns and 'score_high_full' in rw.columns and 'score_low_full' in rw.columns:
    rw['score_diff'] = rw['score_high_full'] - rw['score_low_full']

    by_k = rw.groupby('k')['score_diff'].mean().reset_index()
    plt.figure(figsize=FIG_SIZE)
    sns.barplot(data=by_k, x='k', y='score_diff', color='#84C7F9')
    plt.xlabel(
(SAT Rewrite) (k)
)
    plt.ylabel('Mean high-low score difference')
    save_fig('score_diff_by_k.pdf')

        plt.xlabel("(SAT Rewrite) (k')")
        plt.ylabel('Mean high-low score difference')
        save_fig('score_diff_by_k.pdf')
    if 'low_SES' in rw.columns:
        plt.figure(figsize=(9, 5))
        sns.lineplot(data=rw.groupby(['k', 'low_SES'])['score_diff'].mean().reset_index(), x='k', y='score_diff', hue='low_SES', marker='o')
        plt.xlabel('k')
        plt.ylabel('Mean high-low score difference')
        save_fig('score_diff_by_k_by_ses.pdf')

    # Change vs k=0 by essay
    base = rw[rw['k'] == 0][['essay_id', 'score_diff']].rename(columns={'score_diff': 'score_diff_k0'})
    rw2 = rw.merge(base, on='essay_id', how='left')
    rw2['delta_vs_k0'] = rw2['score_diff'] - rw2['score_diff_k0']

    plt.figure(figsize=(9, 5))
    sns.lineplot(data=rw2.groupby(['k', 'low_SES'])['delta_vs_k0'].mean().reset_index(), x='k', y='delta_vs_k0', hue='low_SES', marker='o')
    plt.axhline(0, color='gray', linestyle='--', linewidth=1)
    plt.xlabel('k')
    plt.ylabel('Mean score_diff - score_diff(k=0)')
    save_fig('score_diff_vs_k0_by_ses.pdf')

In [ ]:
# Mean predicted scores by k x SES with 95% CI
if all(c in rw.columns for c in ['k', 'low_SES', 'score_high_full']):
    grp = rw.groupby(['k', 'low_SES'])['score_high_full'].agg(['mean', 'count', 'std']).reset_index()
    grp['se'] = grp['std'] / np.sqrt(grp['count'].clip(lower=1))
    grp['ci95'] = 1.96 * grp['se']

    plt.figure(figsize=(10, 5))
    sns.pointplot(data=grp, x='k', y='mean', hue='low_SES', dodge=0.35, join=False, errorbar=None)

    for ses in sorted(grp['low_SES'].dropna().unique()):
        s = grp[grp['low_SES'] == ses]
        x = s['k'].to_numpy(dtype=float) + (-0.12 if ses == 0 else 0.12)
        plt.errorbar(x, s['mean'], yerr=s['ci95'], fmt='none', ecolor='black', elinewidth=1, capsize=3)

    plt.xlabel('k')
    plt.ylabel('Mean predicted high score')
    save_fig('rewrite_k_mean_all_CI.pdf')

### Pairwise SES delta analysis (bootstrap CI)

In [ ]:
if all(c in rw.columns for c in ['essay_id', 'k', 'low_SES', 'score_high_full']):
    wide = rw.pivot_table(index=['essay_id', 'low_SES'], columns='k', values='score_high_full', aggfunc='mean')
    ks = [k for k in sorted([c for c in wide.columns if pd.notna(c)]) if k in K_LABELS]

    rows = []
    rng = np.random.default_rng(42)

    for i in range(len(ks)):
        for j in range(i + 1, len(ks)):
            k1, k2 = ks[i], ks[j]
            tmp = wide[[k1, k2]].dropna().reset_index()
            if len(tmp) < 10:
                continue

            d = (tmp[k2] - tmp[k1]).to_numpy()
            point = float(np.mean(d))

            boots = []
            for _ in range(300):
                idx = rng.integers(0, len(d), len(d))
                boots.append(float(np.mean(d[idx])))

            lo, hi = np.percentile(boots, [2.5, 97.5])
            rows.append({
                'k_from': int(k1),
                'k_to': int(k2),
                'delta_mean': point,
                'ci_2_5': float(lo),
                'ci_97_5': float(hi),
                'n': int(len(d))
            })

    pair_df = pd.DataFrame(rows)
    pair_df['pair_label'] = pair_df['k_from'].map(K_LABELS) + ' -> ' + pair_df['k_to'].map(K_LABELS)
    pair_df.to_csv(TABLE_DIR / 'ses_pairwise_diff_values.csv', index=False)

    if len(pair_df):
        plt.figure(figsize=(10, 5))
        sns.histplot(pair_df['delta_mean'], bins=15, kde=True, color='#84C7F9')
        plt.axvline(pair_df['delta_mean'].mean(), color='black', linestyle='--', linewidth=1)
        plt.xlabel('Pairwise mean deltas')
        save_fig('ses_pairwise_diff_distribution_15_with_mean.pdf')

        heat = pair_df.pivot(index='k_from', columns='k_to', values='delta_mean')
        plt.figure(figsize=(8, 6))
        sns.heatmap(heat, cmap='coolwarm', center=0, annot=True, fmt='.2f')
        plt.xlabel('k_to')
        plt.ylabel('k_from')
        save_fig('ses_pairwise_diff_heatmap_upper.pdf')

    pair_df.head()
else:
    print('Skipped pairwise analysis: missing required columns.')

### FE/style distributions and decomposition shares

In [ ]:
if 'fe_essay' in df_decomp.columns:
    plt.figure(figsize=FIG_SIZE)
    sns.histplot(df_decomp['fe_essay'].dropna(), bins=40, color='#84C7F9')
    plt.xlabel('Essay fixed effects')
    save_fig('fixed_effect_distribution.pdf')

if all(c in df_decomp.columns for c in ['fe_essay', 'low_SES']):
    plt.figure(figsize=FIG_SIZE)
    sns.kdeplot(data=df_decomp, x='fe_essay', hue='low_SES', fill=True, common_norm=False, alpha=0.3)
    plt.xlabel('Essay fixed effects')
    save_fig('fixed_effect_distribution_by_SES_dual.pdf')

if 'u' in df_decomp.columns:
    plt.figure(figsize=FIG_SIZE)
    sns.histplot(df_decomp['u'].dropna(), bins=40, color='#C7B6F7')
    plt.xlabel('Style residual u')
    save_fig('style_distribution.pdf')

if all(c in df_decomp.columns for c in ['u', 'low_SES']):
    plt.figure(figsize=FIG_SIZE)
    sns.kdeplot(data=df_decomp, x='u', hue='low_SES', fill=True, common_norm=False, alpha=0.3)
    plt.xlabel('Style residual u')
    save_fig('style_distribution_by_SES_dual.pdf')

In [ ]:
# Decomposition shares (all + optional subgroup breakdowns)
def compute_share_df(df):
    req = {'total_gap', 'content_gap', 'style_gap', 'others_gap'}
    if not req.issubset(df.columns):
        return pd.DataFrame()

    out = df[['total_gap', 'content_gap', 'style_gap', 'others_gap']].copy()
    out = out.replace([np.inf, -np.inf], np.nan).dropna()
    if len(out) == 0:
        return pd.DataFrame()

    out['share_content'] = out['content_gap'] / out['total_gap']
    out['share_style'] = out['style_gap'] / out['total_gap']
    out['share_others'] = out['others_gap'] / out['total_gap']
    return out


share_df = compute_share_df(df_decomp)
if len(share_df):
    means = share_df[['share_content', 'share_style', 'share_others']].mean()
    plot_df = pd.DataFrame([means], index=['All'])

    ax = plot_df.plot(kind='bar', stacked=True, figsize=(7, 4),
                      color=['#84C7F9', '#C7B6F7', '#B6E3C7'], edgecolor='white', linewidth=0.6)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Share of total gap')
    ax.set_xlabel('')
    ax.legend(['Content', 'Style', 'Other'], loc='center left', bbox_to_anchor=(1.01, 0.5), frameon=False)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'decomp_shares_all.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    summary = means.rename_axis('component').reset_index(name='mean_share')
    summary.to_csv(TABLE_DIR / 'decomp_shares_summary.csv', index=False)
    summary
else:
    print('Skipped decomposition shares: expected gap columns not present in df_decomp.')

In [ ]:
print('Unified notebook finished.')
print('Figures:', FIG_DIR)
print('Tables :', TABLE_DIR)